# 第 7 章の講義と実習　比較の可視化

準備セルを実行し，講義では本文の例を手元で確かめます。後半の約 20 分は，条件や問いを変えて自分で分析します。

## 準備

翼が長い個体は，体重も重いでしょうか。本文と同じペンギンの記録を使い，灰色の散布図を，3 種類のペンギンを見分けられる図に描き直します。

Colab では「ドライブにコピーを保存」で自分用のノートを作り，上から準備セルを実行します。CSV は公開 URL から読み込むため，アップロードは不要です。ローカルでは，実習 ZIP を展開してノートブックと CSV を同じフォルダーに置くと，手元のファイルを使えます。

準備セルを実行し，3 種類のペンギンで計 342 件の測定を使うことを確かめます。本文と同じく，四つの測定値がそろう行を使います。

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

file_name = "penguins.csv"
csv_path = Path("data") / file_name
if not csv_path.is_file():
    csv_path = Path("../data") / file_name
if not csv_path.is_file():
    csv_path = Path(file_name)
if not csv_path.is_file():
    csv_path = "https://raw.githubusercontent.com/sohei-takagi/python-ds-course-materials/main/data/" + file_name

penguins = pd.read_csv(csv_path)
has_measurements = (
    penguins["body_mass_g"].notna()
    & penguins["bill_length_mm"].notna()
    & penguins["bill_depth_mm"].notna()
    & penguins["flipper_length_mm"].notna()
)
measured = penguins.loc[has_measurements]
measured.groupby("species").size()

## 講義中に実行するセル

見出しの順に実行します。セルを動かす前に，比べる行・値・図を確認します。

### 1. 図に入れる測定記録を確かめる

準備で，体重・くちばしの長さと厚み・翼の長さがそろう記録を `measured` に入れました。先頭の行で，どの値の組が一羽の記録かを確かめます。

In [ ]:
print("Rows in CSV:", len(penguins), "Rows used:", len(measured))
measured[["species", "body_mass_g", "bill_length_mm", "bill_depth_mm"]].head(3)

### 2. ペンギンの種類ごとに平均体重を比べる

横軸はペンギンの種類，縦軸は平均体重です。高さが近い二つの棒を見つけ，図の元になった記録数と平均を表で確かめます。

In [ ]:
lecture_masses = measured.groupby("species")["body_mass_g"]
lecture_mean_mass = lecture_masses.mean()
lecture_masses.agg(records="size", mean_body_mass_g="mean").round(1)

In [ ]:
fig, ax = plt.subplots(layout="constrained")
ax.bar(lecture_mean_mass.index, lecture_mean_mass, color="#586c7a")
ax.set(xlabel="Species", ylabel="Mean body mass (g)", ylim=(0, 5500))
plt.show()

### 3. 一羽ずつの体重の分布を比べる

各図を同じ区間幅・同じ軸で描きます。平均が近い Adelie と Chinstrap の分布はどこで重なりますか。Gentoo の棒が多い区間も探します。

In [ ]:
fig, axes = plt.subplots(3, 1, sharex=True, sharey=True, figsize=(6.2, 5.8),
                         layout="constrained")
for ax, (species, values) in zip(axes, lecture_masses):
    ax.hist(values, bins=range(2500, 6501, 250), color="#586c7a", edgecolor="white")
    ax.set_title(f"{species} (n = {len(values)})")
axes[-1].set(xlabel="Body mass (g)", xlim=(2500, 6500), ylim=(0, 35))
fig.supylabel("Records")
plt.show()

### 4. 一羽の二つの値を一点にする

横軸にくちばしの長さ，縦軸に厚みを置きます。行ラベル 0 の二つの値を表から読み，図の点線をたどって一つの点を確かめます。

In [ ]:
lecture_bills = measured.loc[[0, 152, 276], ["species", "bill_length_mm", "bill_depth_mm"]]
lecture_bills

In [ ]:
fig, ax = plt.subplots(layout="constrained")
ax.scatter(lecture_bills["bill_length_mm"], lecture_bills["bill_depth_mm"], s=55)
ax.axvline(lecture_bills.loc[0, "bill_length_mm"], color="#586c7a", linestyle=":")
ax.axhline(lecture_bills.loc[0, "bill_depth_mm"], color="#586c7a", linestyle=":")
ax.set(xlabel="Bill length (mm)", ylabel="Bill depth (mm)", xlim=(30, 62), ylim=(12, 22))
plt.show()

### 5. 軸をそろえてペンギンの種類を見分ける

二枚とも同じ 342 件を描きます。右では，ペンギンの種類ごとに色と記号を変えます。横軸の 44〜46 mm のあたりで，厚みの異なる点がどのペンギンの種類か比べます。

In [ ]:
fig, (plain, grouped) = plt.subplots(1, 2, sharex=True, sharey=True,
                                      figsize=(8, 3.6), layout="constrained")
plain.scatter(measured["bill_length_mm"], measured["bill_depth_mm"],
              color="#687c8a", s=15, alpha=0.6)
plain.set(title="All records", ylabel="Bill depth (mm)")
for species, color, marker in zip(
    ["Adelie", "Chinstrap", "Gentoo"], ["#245d8e", "#aa4b24", "#2e745e"], ["o", "s", "^"],
):
    rows = measured.loc[measured["species"] == species]
    grouped.scatter(rows["bill_length_mm"], rows["bill_depth_mm"],
                    label=species, color=color, marker=marker, s=16, alpha=0.7)
grouped.set(title="By species", xlim=(30, 62), ylim=(12, 22))
grouped.legend(frameon=False, fontsize=8)
fig.supxlabel("Bill length (mm)")
plt.show()

## 自分で分析する実習（約 20 分）

### 1. 全体の図を読む（約 5 分）

下の図では，横軸が翼の長さ（mm），縦軸が体重（g）です。一つの点は，同じ行の二つの測定値から作ります。まず，横軸の左側と右側で，体重の値がどのあたりに集まっているかを比べます。

In [ ]:
fig, ax = plt.subplots(layout="constrained")
ax.scatter(measured["flipper_length_mm"], measured["body_mass_g"],
           color="#526678", s=27, alpha=0.60)
ax.set(xlabel="Flipper length (mm)", ylabel="Body mass (g)",
       xlim=(170, 235), ylim=(2500, 6500))
plt.show()

灰色の図を見て，次の二つを一文ずつ書きます。

- 全記録をまとめると，翼が長い側では体重にどんな傾向が見えますか。
- ペンギンの種類の表示を加えたら，この点の集まりについて何を調べたいですか。

（ここに記入）

### 2. ペンギンの種類が分かる図にする（約 9 分）

本文のくちばしの散布図と同じく，一つの `ax` に 3 種類のペンギンを順に描きます。上の図と同じ 342 件を使い，横と縦の表示範囲も揃えます。

下のセルでは，点の色と形を決める設定と，作図の下書きを用意しました。作図の行の先頭の `#` と，その直後の空白 1 個を外し，`scatter()` の `group[""]` 二つへ列名を，`set()` の `xlabel` と `ylabel` へ単位を含む軸名を入れます。`for` の中の行には，繰り返す範囲を示す字下げを残します。

| 何を決めるか | 参照する場所 |
|---|---|
| 横軸と縦軸へ渡す列 | 1 の `scatter()` に渡した二つの列 |
| 横軸と縦軸の名前・単位 | 1 の `set()` で表示した軸名 |
| どのペンギンの種類を描くか | その回に選んだ `group` と `name` |

作図行がコメントの間は，描き直した図は表示されません。

In [ ]:
species_names = ["Adelie", "Chinstrap", "Gentoo"]
colors = {"Adelie": "#286b70", "Chinstrap": "#b15334", "Gentoo": "#695498"}
markers = {"Adelie": "o", "Chinstrap": "^", "Gentoo": "s"}

# fig, ax = plt.subplots(layout="constrained")
# for name in species_names:
#     group = measured.loc[measured["species"] == name]
#     ax.scatter(group[""], group[""], label=name,
#                color=colors[name], marker=markers[name], s=30, alpha=0.65)
# ax.set(xlabel="", ylabel="", xlim=(170, 235), ylim=(2500, 6500))
# ax.legend(loc="upper left")
# plt.show()

灰色の図と，描き直した図を順に見ます。座標や件数を揃えてあるので，同じ位置に，どのペンギンの種類の色や形が加わったかを読めます。本文で，長さが近い点を厚みで見比べたときと同じ読み方です。

<details>
<summary>列名と軸名の書き方を確認する</summary>

横軸に `flipper_length_mm`，縦軸に `body_mass_g` を使います。繰り返しの内側と外側で，次のように書けます。

```python
    ax.scatter(group["flipper_length_mm"], group["body_mass_g"], label=name,
               color=colors[name], marker=markers[name], s=30, alpha=0.65)
ax.set(xlabel="Flipper length (mm)", ylabel="Body mass (g)",
       xlim=(170, 235), ylim=(2500, 6500))
```

</details>

### 3. 描き直して分かったこと（約 6 分）

下の三つを含めて，図から読み取ったことを 2〜3 文で説明します。

1. 最初に書いた疑問への答えとして，ペンギンの種類を表示して新しく分かった特徴を一つ挙げる。どのペンギンの種類の点が，どちらの軸のどのあたりにあるかを書く。
2. ペンギンの種類を表示しても点が重なる範囲を一つ挙げる。
3. 図が対象にした調査の時期と，点の件数を記す。

（ここに記入）

提出前にカーネルを再起動し，上から全セルを実行します。灰色の図とペンギンの種類が分かる図，記入した説明がそろったノートブックを保存します。

#### 出典

2007〜2009 年に南極半島付近の 3 島で調査された成鳥の記録です。Kristen Gorman と Palmer Station LTER の研究者による調査から，Horst，Hill，Gorman が整理・配布した表を使用しています。[データの配布元](https://allisonhorst.github.io/palmerpenguins/) ・ [原著](https://doi.org/10.1371/journal.pone.0090081)